In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [ ]:
# 데이터 불러오기
abalone_df = pd.read_csv(
    'https://storage.googleapis.com/download.tensorflow.org/data/abalone_train.csv',
    names=['Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight',
           'Viscera weight', 'Shell weight', 'Age']
)

# 입력과 타깃 나누기
input_data = abalone_df.drop(columns=['Age']).to_numpy().astype(np.float32)
target_data = abalone_df['Age'].to_numpy().astype(np.float32)

# 데이터셋 클래스 정의
class AbaloneDataset(Dataset):
    def __init__(self, input_data, target_data):
        self.input_data = input_data
        self.target_data = target_data

    def __len__(self):
        return len(self.input_data)

    def __getitem__(self, index):
        input_tensor = torch.tensor(self.input_data[index])
        target_tensor = torch.tensor(self.target_data[index])

        return input_tensor, target_tensor

# 학습/검증/테스트 분할
train_size = int(len(input_data) * 0.8)
val_size = int(len(input_data) * 0.1)

train_inputs = input_data[:train_size]
train_targets = target_data[:train_size]

val_inputs = input_data[train_size:train_size + val_size]
val_targets = target_data[train_size:train_size + val_size]

test_inputs = input_data[train_size + val_size:]
test_targets = target_data[train_size + val_size:]

# 표준화
scaler = StandardScaler()
scaler.fit(train_inputs)

train_inputs_scaled = scaler.transform(train_inputs)
val_inputs_scaled = scaler.transform(val_inputs)
test_inputs_scaled = scaler.transform(test_inputs)

# 데이터셋 만들기
train_dataset = AbaloneDataset(train_inputs_scaled, train_targets)
val_dataset = AbaloneDataset(val_inputs_scaled, val_targets)
test_dataset = AbaloneDataset(test_inputs_scaled, test_targets)

# 데이터로더 만들기
train_dataloader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=32)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=32)

In [ ]:
# 모델 클래스 정의
class AbaloneModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(7, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 8)
        self.fc4 = nn.Linear(8, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.fc4(x)
        return x

'''
Dropout은 Batch Normalization과 더불어서 학습 때와 추론 때 동작이 달라져야 하는 레이어
Dropout 레이어는 학습 때 일부 뉴런을 비활성화하고 추론 때는 모든 뉴런 사용
학습 과정인지 추론 과정인지 알려줘야 해서 throw
'''

# 모델, 손실 함수, 옵티마이저 객체 만들기
model = AbaloneModel()
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters())

'''
GPU 이용
torch.cude.is_available이 True면 CPU, 아니면 GPU'''
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# training loop
epochs = 10
step = 0
for epoch in range(epochs):
    ''' catch : 이 모델은 학습이다 = 일부 뉴론 비활성'''
    model.train()
    for train_batch in train_dataloader:
        x_train, y_train = train_batch[0].to(device), train_batch[1].to(device)
        pred = model(x_train).squeeze()
        loss = loss_fn(pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        #배치마다 step을 1씩을 늘려서 일정한 주기마다 동작 트리거 가
        step += 1

        if step % 100 == 0:
            print(f'step {step}, train loss: {loss.item():.4f}')

    ''' 평가 코드 직전에 메소드 호출로,
    추론 과정에 맞춰 동작해서 Dropout 레이어의 효과가 사라지고 모든 뉴런 활성화 '''
    model.eval()

    # 모델 성능은 epoch 하나가 끝날때마다 함
    with torch.no_grad():
        losses = []
        for val_batch in val_dataloader:
            x_val, y_val = val_batch[0].to(device), val_batch[1].to(device)
            pred = model(x_val).squeeze()
            loss = loss_fn(pred, y_val)

            #loss 발생시마다 리스트에 추가
            losses.append(loss.item())

        val_loss_avg = sum(losses) / len(losses)
        print(f'epoch {epoch+1}/{epochs}, validation loss: {val_loss_avg:.4f}\n')

epoch 1/10, validation loss: 15.5769

step 100, train loss: 30.5581
epoch 2/10, validation loss: 9.7594

step 200, train loss: 22.1156
epoch 3/10, validation loss: 10.7562

step 300, train loss: 22.9992
epoch 4/10, validation loss: 7.0673

step 400, train loss: 15.9458
epoch 5/10, validation loss: 8.3256

epoch 6/10, validation loss: 7.0800

step 500, train loss: 17.1848
epoch 7/10, validation loss: 7.6008

step 600, train loss: 13.1846
epoch 8/10, validation loss: 6.4604

step 700, train loss: 25.6633
epoch 9/10, validation loss: 6.2038

step 800, train loss: 18.4888
epoch 10/10, validation loss: 6.4894



#### 회귀 모델 학습시키기 실습 2

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# 데이터 불러오기
cal_housing = datasets.fetch_california_housing()
input_data = cal_housing.data.astype(np.float32)
target_data = cal_housing.target.astype(np.float32)

# CaliforniaHousingDataset 클래스 정의
class CaliforniaHousingDataset(Dataset):
    def __init__(self, input_data, target_data):
        self.input_data = input_data
        self.target_data = target_data

    def __len__(self):
        return len(self.input_data)

    def __getitem__(self, index):
        input_tensor = torch.tensor(self.input_data[index])
        target_tensor = torch.tensor(self.target_data[index])
        return input_tensor, target_tensor

# 학습/검증/테스트 분할
train_size = int(len(input_data) * 0.8)
val_size = int(len(input_data) * 0.1)

train_inputs = input_data[:train_size]
train_targets = target_data[:train_size]

val_inputs = input_data[train_size:train_size + val_size]
val_targets = target_data[train_size:train_size + val_size]

test_inputs = input_data[train_size + val_size:]
test_targets = target_data[train_size + val_size:]

# 학습 입력 데이터 기준 표준화
scaler = StandardScaler()
scaler.fit(train_inputs)

train_inputs_scaled = scaler.transform(train_inputs)
val_inputs_scaled = scaler.transform(val_inputs)
test_inputs_scaled = scaler.transform(test_inputs)

# Dataset 객체 생성
train_dataset = CaliforniaHousingDataset(train_inputs_scaled, train_targets)
val_dataset = CaliforniaHousingDataset(val_inputs_scaled, val_targets)
test_dataset = CaliforniaHousingDataset(test_inputs_scaled, test_targets)

# DataLoader 객체 생성
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=4)
test_dataloader = DataLoader(test_dataset, batch_size=4)

# CaliforniaHousingModel 클래스 정의
class CaliforniaHousingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear0 = nn.Linear(8, 16)
        self.linear1 = nn.Linear(16, 32)
        self.linear2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.linear0(x))
        x = self.relu(self.linear1(x))
        output = self.linear2(x)
        return output

# 모델 객체 생성
model = CaliforniaHousingModel()

# loss function, optimizer 객체 생성
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters())





# training loop
epochs = 10
step = 0
for epoch in range(epochs):
    for train_batch in train_dataloader:
        x_train, y_train =  train_batch
        pred = model(x_train).squeeze()
        loss = loss_fn(pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        step += 1
        if step % 1000 == 0:
            print(f'step {step}, train loss: {loss.item():.4f}')
        # 여기에 코드를 작성하세요(학습 loss 프린트)

    # 여기에 코드를 작성하세요(모델 성능 평가 및 결과 프린트).
    model.eval()

    # 모델 성능은 epoch 하나가 끝날때마다 함
    with torch.no_grad():
        losses = []
        for val_batch in val_dataloader:
            x_val, y_val =  val_batch
            pred = model(x_val).squeeze()
            loss = loss_fn(pred, y_val)

            #loss 발생시마다 리스트에 추가
            losses.append(loss.item())

        val_loss_avg = sum(losses) / len(losses)
        print(f'epoch {epoch+1}, avg val loss: {val_loss_avg:.4f}\n')


step 1000, train loss: 0.4637
epoch 1, avg val loss: 0.6396

step 2000, train loss: 0.8446
epoch 2, avg val loss: 0.5660

step 3000, train loss: 0.4186
epoch 3, avg val loss: 0.5851

step 4000, train loss: 0.4053
epoch 4, avg val loss: 0.5502

step 5000, train loss: 0.2335
epoch 5, avg val loss: 0.6371

step 6000, train loss: 0.3306
epoch 6, avg val loss: 0.5741

step 7000, train loss: 0.4257
epoch 7, avg val loss: 0.5508

step 8000, train loss: 0.4499
epoch 8, avg val loss: 0.5602

step 9000, train loss: 0.1240
epoch 9, avg val loss: 0.5856

step 10000, train loss: 0.0694
epoch 10, avg val loss: 0.5748



#### 회귀 모델 학습시키기 실습 3

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# 데이터 불러오기
cal_housing = datasets.fetch_california_housing()
input_data = cal_housing.data.astype(np.float32)
target_data = cal_housing.target.astype(np.float32)

# CaliforniaHousingDataset 클래스 정의
class CaliforniaHousingDataset(Dataset):
    def __init__(self, input_data, target_data):
        self.input_data = input_data
        self.target_data = target_data

    def __len__(self):
        return len(self.input_data)

    def __getitem__(self, index):
        input_tensor = torch.tensor(self.input_data[index])
        target_tensor = torch.tensor(self.target_data[index])
        return input_tensor, target_tensor

# 학습/검증/테스트 분할
train_size = int(len(input_data) * 0.8)
val_size = int(len(input_data) * 0.1)

train_inputs = input_data[:train_size]
train_targets = target_data[:train_size]

val_inputs = input_data[train_size:train_size + val_size]
val_targets = target_data[train_size:train_size + val_size]

test_inputs = input_data[train_size + val_size:]
test_targets = target_data[train_size + val_size:]

# 학습 입력 데이터 기준 표준화
scaler = StandardScaler()
scaler.fit(train_inputs)

train_inputs_scaled = scaler.transform(train_inputs)
val_inputs_scaled = scaler.transform(val_inputs)
test_inputs_scaled = scaler.transform(test_inputs)

# Dataset 객체 생성
train_dataset = CaliforniaHousingDataset(train_inputs_scaled, train_targets)
val_dataset = CaliforniaHousingDataset(val_inputs_scaled, val_targets)
test_dataset = CaliforniaHousingDataset(test_inputs_scaled, test_targets)

# DataLoader 객체 생성
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=4)
test_dataloader = DataLoader(test_dataset, batch_size=4)

# CaliforniaHousingModel 클래스 정의(Dropout 포함)
class CaliforniaHousingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear0 = nn.Linear(8, 16)
        self.linear1 = nn.Linear(16, 32)
        self.linear2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout()

    def forward(self, x):
        x = self.relu(self.linear0(x))
        x = self.relu(self.linear1(x))
        x = self.dropout(x)
        output = self.linear2(x)
        return output

# 모델 객체 생성
model = CaliforniaHousingModel()

# loss function, optimizer, device 객체 생성
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 여기에 코드를 작성하세요(모델 연산 장치 설정).
model.to(device)

# training loop
epochs = 10
step = 0
for epoch in range(epochs):
    # 여기에 코드를 작성하세요(모델 학습 과정 설정).
    model.train()
    for train_batch in train_dataloader:
        x_train, y_train = train_batch[0].to(device), train_batch[1].to(device)
        pred = model(x_train).squeeze()
        loss = loss_fn(pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        step += 1
        if step % 1000 == 0:
            print(f'step {step}, train loss: {loss.item():.4f}')

    # 여기에 코드를 작성하세요(모델 평가 과정 설정).
    model.eval()
    with torch.no_grad():
        losses = []
        for val_batch in val_dataloader:
            x_val, y_val = val_batch[0].to(device), val_batch[1].to(device)
            pred = model(x_val).squeeze()
            loss = loss_fn(pred, y_val)
            losses.append(loss.item())

    val_loss_avg = sum(losses) / len(losses)
    print(f'epoch {epoch+1}, val loss {val_loss_avg:.4f}\n')

step 1000, train loss: 1.5167
epoch 1, val loss 0.8784

step 2000, train loss: 1.2112
epoch 2, val loss 0.6118

step 3000, train loss: 0.2516
epoch 3, val loss 0.5871

step 4000, train loss: 0.3074
epoch 4, val loss 0.6206

step 5000, train loss: 1.3389
epoch 5, val loss 0.6470

step 6000, train loss: 0.1895
epoch 6, val loss 0.7077

step 7000, train loss: 0.1574
epoch 7, val loss 0.6346

step 8000, train loss: 0.6140
epoch 8, val loss 0.6623

step 9000, train loss: 0.5169
epoch 9, val loss 0.6776

step 10000, train loss: 0.1066
epoch 10, val loss 0.6319

